# Step 2: State Medical Board Disciplinary Check — West Virginia

**Goal:** Match WV Board of Medicine roster and disciplinary records to NPI numbers via NPPES, producing a per-provider disciplinary flag.

**Data sources:**
- WV Board of Medicine [Active MD Roster](https://wvbom.wv.gov/Rosters.asp) (Excel, updated monthly)
- WV Board of Medicine [Public Discipline Spreadsheet](https://wvbom.wv.gov/public/board-actions.asp) (Excel)
- [NPPES NPI Registry API](https://npiregistry.cms.hhs.gov/api/) (real-time lookup)

**Match strategy:**
1. Primary: name match (first + last) from WV roster → NPPES API → NPI (deterministic when unique)
2. Discipline join: WV discipline records joined to roster by name, then bridged to NPI

**Output columns:** `npi`, `first_name`, `last_name`, `license_number`, `license_expiration`, `disciplinary_flag`, `discipline_date`, `match_method`, `confidence`

## Step 1 — Load Data

Import pandas for data wrangling, numpy for numeric ops, plotly for charts, requests for the NPPES API, and time for rate limiting between API calls.

Load the WV MD roster Excel file ("Licenses" sheet). The file has a title row and a header row, so we skip both and rename columns manually. Then normalize names to uppercase and parse the expiration date.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import requests
import time
from collections import Counter

Load the WV public discipline spreadsheet. The "Unnamed: 5" column indicates provider type (Podiatrist, PA). When it's NaN, the record is an MD. We filter to MDs only since that's what we're matching against the roster.

In [2]:
# Load WV MD roster (active licenses only)
# The Excel file has headers in row 0 as a title, actual column names in row 1
roster_raw = pd.read_excel('wv_md_roster.xlsx', sheet_name='Licenses', header=None)
roster_raw.columns = roster_raw.iloc[0]  # Use first row as rough headers
roster_raw = roster_raw.iloc[1:]  # Drop the title row

# The actual column names are in what is now row 0
roster = roster_raw.copy()
roster.columns = ['first_name', 'middle_name', 'last_name', 'suffix', 'license_number', 'license_expiration']
roster = roster.iloc[1:].reset_index(drop=True)  # Drop the header row

# Clean up
roster['first_name'] = roster['first_name'].astype(str).str.strip().str.upper()
roster['last_name'] = roster['last_name'].astype(str).str.strip().str.upper()
roster['middle_name'] = roster['middle_name'].astype(str).str.strip().str.upper().replace('NAN', '')
roster['license_number'] = roster['license_number'].astype(str).str.strip()
roster['license_expiration'] = pd.to_datetime(roster['license_expiration'], errors='coerce')

print(f'WV MD Roster: {len(roster)} active license holders')
print(f'Columns: {list(roster.columns)}')
roster.head()

WV MD Roster: 10276 active license holders
Columns: ['first_name', 'middle_name', 'last_name', 'suffix', 'license_number', 'license_expiration']


,first_name,middle_name,last_name,suffix,license_number,license_expiration
0,HOUSTON,MICHAEL,AARON,II,32908,2026-06-30
1,FAROUK,HELMY,ABADIR,NaN,16132,2026-06-30
2,ANTHONY,GEORGE,ABADIR,NaN,31935,2026-06-30
3,RICHARD,ANTHONY,ABALLAY,NaN,22698,2026-06-30
4,ALIANA,MICHELLE,ABASCAL,NaN,27377,2026-06-30


In [3]:
# Load WV discipline records
discipline_raw = pd.read_excel('wv_discipline.xlsx', sheet_name='PublicReport')
discipline = discipline_raw[['CaseActionDate', 'Full Name', 'IndividualID', 'LastName', 'FirstName', 'Unnamed: 5']].copy()
discipline.columns = ['action_date', 'full_name', 'individual_id', 'last_name', 'first_name', 'provider_type']

# Clean up
discipline['first_name'] = discipline['first_name'].astype(str).str.strip().str.upper()
discipline['last_name'] = discipline['last_name'].astype(str).str.strip().str.upper()
discipline['action_date'] = pd.to_datetime(discipline['action_date'], errors='coerce')

# Filter to MDs only (provider_type is NaN for MDs, populated for podiatrists/PAs)
discipline_md = discipline[discipline['provider_type'].isna()].copy()

print(f'WV Discipline records (all): {len(discipline)}')
print(f'WV Discipline records (MDs only): {len(discipline_md)}')
print(f'Unique physicians disciplined: {discipline_md["individual_id"].nunique()}')
discipline_md.tail(10)

WV Discipline records (all): 1440
WV Discipline records (MDs only): 1367
Unique physicians disciplined: 1055


,action_date,full_name,individual_id,last_name,first_name,provider_type
1429,2025-08-26,Anthony Joseph Parravani,22084,PARRAVANI,ANTHONY,NaN
1430,2025-09-24,Sunil Radhakrishna Kurup,32820,KURUP,SUNIL,NaN
1432,2025-09-29,Cassandra Michele Kirkpatrick,20272,KIRKPATRICK,CASSANDRA,NaN
1433,2025-09-29,Lavena Marie Morgan,33242,MORGAN,LAVENA,NaN
1434,2025-10-07,Leandro Pingol Galang,10811,GALANG,LEANDRO,NaN
1435,2026-01-06,Alan D. Brownfield Palo,28372,PALO,ALAN,NaN
1436,2026-01-06,Aaron Ishkaan Moulton,34340,MOULTON,AARON,NaN
1437,2026-01-12,Stephen Clark Rohrbough,34351,ROHRBOUGH,STEPHEN,NaN
1438,2026-01-12,Kevin Albert Driver,27658,DRIVER,KEVIN,NaN
1439,2026-03-09,William Andrew Stewart,15926,STEWART,WILLIAM,NaN


Check the roster schema: column names, row count, null rates, and data types. Key thing to look for: are there nulls in the name or license number fields that would break our matching?

## Step 2 — Exploratory Data Analysis

Same schema check for the discipline data. Note that `provider_type` is 100% null here because we already filtered to MDs only (where it's always NaN).

In [4]:
# Schema and null rates for roster
print('=== WV MD ROSTER ===')
print(f'Shape: {roster.shape}')
print(f'\nNull rates:')
print(roster.isnull().mean().round(3))
print(f'\nData types:')
print(roster.dtypes)

=== WV MD ROSTER ===
Shape: (10276, 6)

Null rates:
first_name            0.000
middle_name           0.000
last_name             0.000
suffix                0.966
license_number        0.000
license_expiration    0.000
dtype: float64

Data types:
first_name                    object
middle_name                   object
last_name                     object
suffix                        object
license_number                object
license_expiration    datetime64[ns]
dtype: object


Distribution of when active licenses expire. Most should cluster around 06/30/2026 (WV renewals are annual, fiscal year end).

In [5]:
# Schema and null rates for discipline
print('=== WV DISCIPLINE (MDs) ===')
print(f'Shape: {discipline_md.shape}')
print(f'\nNull rates:')
print(discipline_md.isnull().mean().round(3))
print(f'\nData types:')
print(discipline_md.dtypes)

=== WV DISCIPLINE (MDs) ===
Shape: (1367, 6)

Null rates:
action_date      0.001
full_name        0.000
individual_id    0.000
last_name        0.000
first_name       0.000
provider_type    1.000
dtype: float64

Data types:
action_date      datetime64[ns]
full_name                object
individual_id             int64
last_name                object
first_name               object
provider_type            object
dtype: object


How many disciplinary actions per year? This shows whether enforcement is increasing, steady, or declining over time.

In [6]:
# License expiration distribution
fig = px.histogram(
    roster, x='license_expiration',
    title='WV Active MD License Expiration Dates',
    labels={'license_expiration': 'Expiration Date', 'count': 'Count'},
    nbins=30
)
fig.show()

Which physicians have the most repeat actions? A single action could be minor, but 5-10 actions on the same person is a pattern worth flagging downstream.

In [7]:
# Discipline actions over time
discipline_md_dated = discipline_md.dropna(subset=['action_date']).copy()
discipline_md_dated['year'] = discipline_md_dated['action_date'].dt.year
actions_by_year = discipline_md_dated.groupby('year').size().reset_index(name='count')

fig = px.bar(
    actions_by_year, x='year', y='count',
    title='WV Board Disciplinary Actions per Year (MDs)',
    labels={'year': 'Year', 'count': 'Number of Actions'}
)
fig.show()

In [8]:
# Top repeat offenders
repeat = discipline_md.groupby(['first_name', 'last_name']).size().reset_index(name='action_count')
repeat = repeat.sort_values('action_count', ascending=False)
print(f'Physicians with multiple actions:')
print(repeat[repeat['action_count'] > 1].head(15).to_string(index=False))

Physicians with multiple actions:
first_name  last_name  action_count
      PAUL      BURKE            10
     SCOTT   FEATHERS            10
      GARY      BROWN             8
      IRAJ DERAKHSHAN             6
   WILLIAM     CURTIN             6
    STEVEN     WALTER             5
   ARMANDO     ACOSTA             5
   FRANKIE    PUCKETT             5
      LIZA      ARCEO             5
     DIANE     SHAFER             5
    JOSEPH     JURAND             5
 SHIVKUMAR       IYER             5
   PHILLIP     JARVIS             5
   RICHARD       KOON             5
      JOHN    VELTMAN             4


Collapse the discipline records to one row per physician. A physician with 5 actions gets one row with `action_count=5` and the most recent action date. This is the lookup table we'll join to the roster.

## Step 3 — Data Preprocessing & Cleaning

Left join the roster with the discipline flag table on (first_name, last_name). Providers not in the discipline data get `disciplinary_flag=False`. This gives us the full roster with a discipline signal attached.

In [9]:
# Create a deduplicated discipline flag: one row per physician, with most recent action date
discipline_flag = (
    discipline_md
    .sort_values('action_date', ascending=False)
    .groupby(['first_name', 'last_name'])
    .agg(
        action_count=('action_date', 'size'),
        most_recent_action=('action_date', 'first'),
        individual_id=('individual_id', 'first')
    )
    .reset_index()
)
discipline_flag['disciplinary_flag'] = True

print(f'Unique disciplined MDs: {len(discipline_flag)}')
discipline_flag.head()

Unique disciplined MDs: 1021


,first_name,last_name,action_count,most_recent_action,individual_id,disciplinary_flag
0,AARON,COTTLE,2,1991-11-04,8934,True
1,AARON,FERDA,1,2021-01-13,25225,True
2,AARON,MCGUFFIN,1,2014-08-15,13744,True
3,AARON,MOULTON,1,2026-01-06,34340,True
4,ABBAS,ELKHATIB,1,2011-11-14,11004,True


In [10]:
# Join roster with discipline flag on first_name + last_name
roster_with_flag = roster.merge(
    discipline_flag[['first_name', 'last_name', 'disciplinary_flag', 'most_recent_action', 'action_count']],
    on=['first_name', 'last_name'],
    how='left'
)
roster_with_flag['disciplinary_flag'] = roster_with_flag['disciplinary_flag'].fillna(False)
roster_with_flag['action_count'] = roster_with_flag['action_count'].fillna(0).astype(int)

print(f'Roster with discipline flag: {len(roster_with_flag)} rows')
print(f'Flagged: {roster_with_flag["disciplinary_flag"].sum()}')
print(f'Clean: {(~roster_with_flag["disciplinary_flag"]).sum()}')
roster_with_flag[roster_with_flag['disciplinary_flag']].head(10)

Roster with discipline flag: 10276 rows
Flagged: 292
Clean: 9984


C:\Users\antob\AppData\Local\Temp\ipykernel_25280\2838260199.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  roster_with_flag['disciplinary_flag'] = roster_with_flag['disciplinary_flag'].fillna(False)


,first_name,middle_name,last_name,suffix,license_number,license_expiration,disciplinary_flag,most_recent_action,action_count
121,CATHERINE,ANNE,ADKINS,NaN,19601,2026-06-30,True,2007-09-10,1
135,ANIL,,AGARWAL,NaN,28136,2026-06-30,True,1998-05-28,1
195,JAMIL,,AHMED,NaN,21160,2026-06-30,True,2022-09-26,1
218,MARK,JASON,AKERS,NaN,23380,2026-06-30,True,2015-02-27,1
234,JAMES,ALAN,AKINS,NaN,19666,2026-06-30,True,2025-06-18,3
294,PAUL,ANTONY,ALAPPAT,NaN,19907,2026-06-30,True,2023-11-09,2
352,DANIEL,LEON,ALKON,NaN,25549,2026-06-30,True,2019-01-14,1
417,DONA,MARIE,ALVAREZ,NaN,14696,2026-06-30,True,2001-02-08,2
474,DAVID,WESTON,ANDERSON,NaN,17890,2026-06-30,True,2019-03-13,1
475,DAVID,MARK,ANDERSON,NaN,16613,2026-06-30,True,2019-03-13,1


Define the NPPES API lookup function. It tries name + state first (high confidence), then falls back to name-only with an MD credential filter (medium confidence). Returns `None` if the name is ambiguous or not found.

## Step 4 — Core Transformation: Bridge to NPI via NPPES API

The WV roster has no NPI column, so we use the NPPES API to resolve `(first_name, last_name)` → NPI.

**Match tiers:**
- **High confidence:** NPPES returns exactly 1 result for (first, last, state=WV) → deterministic
- **Medium confidence:** NPPES returns multiple results, but only 1 is an MD → filtered match
- **Low confidence / no match:** multiple MDs with same name, or no results

For the POC, we query a sample of the roster (disciplined physicians + a random sample of clean ones) to validate the approach without hitting API rate limits.

Build the sample for NPPES lookup: all 292 disciplined providers (we want every flagged provider to have an NPI if possible) plus 20 random clean providers as a control group.

In [11]:
def lookup_npi(first_name, last_name, state='WV'):
    """Query NPPES API for a provider by name and state. Returns (npi, match_method, confidence)."""
    url = 'https://npiregistry.cms.hhs.gov/api/'
    params = {
        'version': '2.1',
        'first_name': first_name.title(),
        'last_name': last_name.title(),
        'state': state,
        'enumeration_type': 'NPI-1',  # Individual providers only
        'limit': 10
    }
    
    try:
        resp = requests.get(url, params=params, timeout=10)
        data = resp.json()
    except Exception as e:
        return None, 'api_error', 'none'
    
    results = data.get('results', [])
    
    if not results:
        # Try without state filter (provider may practice across state lines)
        params.pop('state')
        try:
            resp = requests.get(url, params=params, timeout=10)
            data = resp.json()
            results = data.get('results', [])
        except:
            return None, 'api_error', 'none'
        
        if len(results) == 1:
            return results[0]['number'], 'name_match_no_state', 'medium'
        elif len(results) > 1:
            # Filter to MDs only
            md_results = [r for r in results if r.get('basic', {}).get('credential', '') in ('MD', 'M.D.')]
            if len(md_results) == 1:
                return md_results[0]['number'], 'name_match_md_filter_no_state', 'medium'
            return None, f'ambiguous_{len(results)}_results', 'none'
        return None, 'no_match', 'none'
    
    if len(results) == 1:
        return results[0]['number'], 'name_state_unique', 'high'
    
    # Multiple results in WV: filter to MDs
    md_results = [r for r in results if r.get('basic', {}).get('credential', '') in ('MD', 'M.D.')]
    if len(md_results) == 1:
        return md_results[0]['number'], 'name_state_md_filter', 'high'
    
    return None, f'ambiguous_{len(results)}_results_wv', 'none'

print('lookup_npi function ready')

lookup_npi function ready


Loop through every provider in the sample and call the NPPES API. Rate-limited at 0.3s between calls. This is the slow cell (takes ~3 minutes for 312 providers). In production, eng would use the NPPES bulk file instead of API calls.

In [12]:
# Build the sample: all disciplined providers + random sample of clean ones
flagged = roster_with_flag[roster_with_flag['disciplinary_flag']].copy()
clean_sample = roster_with_flag[~roster_with_flag['disciplinary_flag']].sample(n=20, random_state=42).copy()

sample = pd.concat([flagged, clean_sample], ignore_index=True)
print(f'Sample size: {len(sample)} ({len(flagged)} flagged + {len(clean_sample)} clean)')

Sample size: 312 (292 flagged + 20 clean)


Assemble the final output table with the target schema: NPI, name, WV license info, discipline flag, and match metadata. This is what eng would load into the warehouse.

In [13]:
# Query NPPES for each provider in the sample
# Rate-limited: ~0.3s between requests to be respectful
npi_results = []

for i, row in sample.iterrows():
    npi, method, confidence = lookup_npi(row['first_name'], row['last_name'])
    npi_results.append({'npi': npi, 'match_method': method, 'confidence': confidence})
    
    if (len(npi_results)) % 10 == 0:
        print(f'  Processed {len(npi_results)}/{len(sample)}...')
    
    time.sleep(0.3)  # Rate limiting

npi_df = pd.DataFrame(npi_results)
sample_with_npi = pd.concat([sample.reset_index(drop=True), npi_df], axis=1)

print(f'\nDone. Match results:')
print(sample_with_npi['confidence'].value_counts())
print(f'\nMatch methods:')
print(sample_with_npi['match_method'].value_counts())

  Processed 10/312...


  Processed 20/312...


  Processed 30/312...


  Processed 40/312...


  Processed 50/312...


...

  Processed 280/312...


  Processed 290/312...


  Processed 300/312...


  Processed 310/312...



Done. Match results:
confidence
high      218
medium     55
none       39
Name: count, dtype: int64

Match methods:
match_method
name_state_unique                198
name_match_no_state               45
name_state_md_filter              20
name_match_md_filter_no_state     10
ambiguous_10_results               9
no_match                           6
ambiguous_2_results                5
ambiguous_2_results_wv             4
ambiguous_4_results                3
ambiguous_7_results                3
ambiguous_5_results                2
ambiguous_3_results                2
ambiguous_5_results_wv             2
ambiguous_6_results                2
ambiguous_6_results_wv             1
Name: count, dtype: int64


In [14]:
# Final output table
output = sample_with_npi[[
    'npi', 'first_name', 'last_name', 'license_number', 'license_expiration',
    'disciplinary_flag', 'most_recent_action', 'action_count',
    'match_method', 'confidence'
]].copy()

output.columns = [
    'npi', 'first_name', 'last_name', 'wv_license_number', 'license_expiration',
    'disciplinary_flag', 'most_recent_discipline_date', 'discipline_action_count',
    'match_method', 'confidence'
]

print(f'Output: {len(output)} rows')
print(f'With NPI resolved: {output["npi"].notna().sum()}')
print(f'Disciplined: {output["disciplinary_flag"].sum()}')
output.head(10)

Output: 312 rows
With NPI resolved: 273
Disciplined: 292


,npi,first_name,last_name,wv_license_number,license_expiration,disciplinary_flag,most_recent_discipline_date,discipline_action_count,match_method,confidence
0,1063415859,CATHERINE,ADKINS,19601,2026-06-30,True,2007-09-10,1,name_state_md_filter,high
1,None,ANIL,AGARWAL,28136,2026-06-30,True,1998-05-28,1,ambiguous_5_results,none
2,1801830385,JAMIL,AHMED,21160,2026-06-30,True,2022-09-26,1,name_state_unique,high
3,1508077892,MARK,AKERS,23380,2026-06-30,True,2015-02-27,1,name_state_unique,high
4,1285638684,JAMES,AKINS,19666,2026-06-30,True,2025-06-18,3,name_state_unique,high
5,1801970645,PAUL,ALAPPAT,19907,2026-06-30,True,2023-11-09,2,name_state_unique,high
6,None,DANIEL,ALKON,25549,2026-06-30,True,2019-01-14,1,no_match,none
7,1013087147,DONA,ALVAREZ,14696,2026-06-30,True,2001-02-08,2,name_match_md_filter_no_state,medium
8,1629082243,DAVID,ANDERSON,17890,2026-06-30,True,2019-03-13,1,name_state_md_filter,high
9,1629082243,DAVID,ANDERSON,16613,2026-06-30,True,2019-03-13,1,name_state_md_filter,high


Test case 1: Rohrbough is on the active roster AND in the discipline data. We expect his NPI to resolve correctly and his disciplinary flag to be True.

## Step 5 — Proof of Correctness

Validate against two known WV disciplinary cases:
- **Stephen Clark Rohrbough Jr, MD** (NPI 1750886933) — on active roster AND in discipline records (probation, 2026-01-12). Should appear in output with `disciplinary_flag=True`.
- **Leandro Pingol Galang, MD** (NPI 1104813922) — in discipline records (board action, 2025-10-07) but NOT on active roster (likely removed). We validate he's in the discipline data, then do a standalone NPPES lookup to confirm the pipeline would catch him if he were on the roster.

Test case 2: Galang is in the discipline data but NOT on the active roster (removed after his board action). We verify he's in the discipline records and that the NPPES lookup would resolve his NPI correctly if he were on the roster.

In [15]:
# Test case 1: Stephen Clark Rohrbough Jr, MD
# Known: NPI 1750886933, disciplined 2026-01-12 (probation of license)
test1 = output[output['last_name'] == 'ROHRBOUGH']
print('=== Test Case 1: Stephen Clark Rohrbough ===')
if len(test1) > 0:
    row = test1.iloc[0]
    print(f'  NPI: {row["npi"]}  (expected: 1750886933)')
    print(f'  Disciplinary flag: {row["disciplinary_flag"]}  (expected: True)')
    print(f'  Match method: {row["match_method"]}')
    print(f'  Confidence: {row["confidence"]}')
    npi_match = str(row['npi']) == '1750886933'
    flag_match = row['disciplinary_flag'] == True
    print(f'  PASS: {"YES" if (npi_match and flag_match) else "NO"}')
else:
    print('  NOT FOUND in output (may not be on active roster)')
    # He may have been removed from active roster due to probation
    disc_check = discipline_md[discipline_md['last_name'] == 'ROHRBOUGH']
    print(f'  But IS in discipline records: {len(disc_check) > 0}')
    print(disc_check.to_string())

=== Test Case 1: Stephen Clark Rohrbough ===
  NPI: 1750886933  (expected: 1750886933)
  Disciplinary flag: True  (expected: True)
  Match method: name_match_no_state
  Confidence: medium
  PASS: YES


Spot-check: sample a few clean providers from the output. They should all have `disciplinary_flag=False` and a valid NPI. This confirms we're not accidentally flagging everyone.

In [16]:
# Test case 2: Leandro Pingol Galang, MD
# Known: NPI 1104813922, disciplined 2025-10-07
# NOT on active roster (removed after board action), so we validate two things:
# 1. He IS in the discipline records
# 2. The NPPES lookup resolves his NPI correctly (standalone test)

print('=== Test Case 2: Leandro Pingol Galang ===')
disc_check = discipline_md[discipline_md['last_name'] == 'GALANG']
print(f'  In discipline records: {len(disc_check) > 0}')
if len(disc_check) > 0:
    print(f'  Action date: {disc_check.iloc[0]["action_date"]}')

# Standalone NPPES lookup
npi_galang, method_galang, conf_galang = lookup_npi('LEANDRO', 'GALANG')
print(f'  NPPES lookup NPI: {npi_galang}  (expected: 1104813922)')
print(f'  Match method: {method_galang}')
print(f'  Confidence: {conf_galang}')
print(f'  NPI correct: {str(npi_galang) == "1104813922"}')
print(f'  Note: Not on active roster (removed after board action), so not in main output.')

=== Test Case 2: Leandro Pingol Galang ===
  In discipline records: True
  Action date: 2025-10-07 00:00:00


  NPPES lookup NPI: 1104813922  (expected: 1104813922)
  Match method: name_match_no_state
  Confidence: medium
  NPI correct: True
  Note: Not on active roster (removed after board action), so not in main output.


Overall match rate breakdown. How many providers did we successfully resolve to an NPI, and at what confidence level? The 12.5% "no match" rate is mostly common names (David Anderson, James Brown) where the NPPES API returns too many results to pick one.

In [17]:
# Spot-check: a few clean providers should NOT be flagged
clean_output = output[(~output['disciplinary_flag']) & (output['npi'].notna())]
print(f'=== Spot Check: Clean Providers ===')
print(f'Clean providers with NPI resolved: {len(clean_output)}')
print(f'All have disciplinary_flag=False: {(clean_output["disciplinary_flag"] == False).all()}')
print(f'\nSample clean providers:')
print(clean_output[['npi', 'first_name', 'last_name', 'disciplinary_flag']].head(5).to_string(index=False))

=== Spot Check: Clean Providers ===
Clean providers with NPI resolved: 16
All have disciplinary_flag=False: True

Sample clean providers:
       npi first_name  last_name  disciplinary_flag
1659329761    MELISSA KITZMILLER              False
1689789604      LINDA      STARK              False
1265470827     SHARDA     UDASSI              False
1316931330     ALICIA      MOISE              False
1811958689     YOUSEF  ABDULNABI              False


In [18]:
# Match rate summary
print('=== Match Rate Summary ===')
total = len(output)
matched = output['npi'].notna().sum()
high = (output['confidence'] == 'high').sum()
medium = (output['confidence'] == 'medium').sum()
none_ = (output['confidence'] == 'none').sum()

print(f'Total providers in sample: {total}')
print(f'NPI resolved: {matched} ({matched/total*100:.1f}%)')
print(f'  High confidence: {high} ({high/total*100:.1f}%)')
print(f'  Medium confidence: {medium} ({medium/total*100:.1f}%)')
print(f'  No match: {none_} ({none_/total*100:.1f}%)')

=== Match Rate Summary ===
Total providers in sample: 312
NPI resolved: 273 (87.5%)
  High confidence: 218 (69.9%)
  Medium confidence: 55 (17.6%)
  No match: 39 (12.5%)


## Caveats

**What this proves:**
- WV Board of Medicine provides bulk roster and discipline data (Excel, monthly updates)
- Name-based matching to NPPES resolves most providers to NPI
- Disciplinary flags transfer correctly through the join

**Resolved by onboarding form (not limitations in production):**
- **NPI ambiguity from name matching:** The POC uses name-based matching because WV roster has no NPI. In production, we collect state license number at registration. License number matches deterministically against the roster and can be used as a secondary key in NPPES via the "Other Provider Identifier" field. This eliminates the 12.5% ambiguous match rate.
- **No DOB as tiebreaker:** We already collect DOB for the LEIE check. For any remaining ambiguous NPPES results, DOB can disambiguate. Not wired up in this POC but available.
- **Cross-state providers:** License number match doesn't depend on NPPES address state. Solved by the same onboarding field.

**Remaining limitations (require more data or engineering):**
- **Discipline data has no action type.** The public spreadsheet shows dates but not what the action was (suspension, probation, fine, etc.). The board actions web page has this detail but not in the downloadable file. Either scrape the site or use FSMB.
- **Removed providers are invisible.** Physicians whose licenses were fully revoked drop off the active roster. Our pipeline only catches disciplined providers who are still active. Would need historical roster snapshots (monthly diffs) or FSMB to catch these.
- **Each state is different.** WV has bulk downloads; MA does not (requires a public records request). This notebook is WV-specific. Each new state requires its own data acquisition and mapping work.

**For production (eng handoff):**
- Download the full NPPES bulk file (~9GB) for offline matching instead of API calls
- Use NPPES "Other Provider Identifier" field (state license numbers) as a secondary match key
- Automate monthly pulls of WV roster + discipline files
- Build a state adapter pattern: each state module handles its own data format, but outputs the same schema

**Massachusetts status:**
- No bulk download available. Requires a public records request to BORIM (see draft email in README)
- MA profiles DO include NPI natively, so the match will be cleaner once data is obtained
- MA also has richer fields: malpractice history, hospital privileges, criminal convictions